# This might be all we need!

In [104]:
from schwingermodel import generateSchwingerHamiltonian

In [105]:
from schwingermodel import exact_ground_state_energy 

In [106]:
from qiskit.quantum_info import SparsePauliOp

def bitflip_projection(op: SparsePauliOp) -> SparsePauliOp:
    terms = []
    for label, coeff in zip(op.paulis.to_labels(), op.coeffs):
        new_label = label.replace("Y", "X").replace("Z", "I")
        terms.append((new_label, coeff))
    return SparsePauliOp.from_list(terms).simplify()


def apply_ix_string_to_bitstring(ix_label: str, bitstring: str) -> str:
    out = list(bitstring)
    for j, p in enumerate(ix_label):
        if p == "X":
            out[j] = "1" if out[j] == "0" else "0"
    return "".join(out)


def reachable_bitstrings_n_steps(op: SparsePauliOp, initial_bitstrings, n_steps: int):
    """
    Ignore amplitudes entirely.
    Track only which bitstrings are reachable after n_steps applications
    of the projected operator's flip masks.
    """
    if isinstance(initial_bitstrings, str):
        current = {initial_bitstrings}
    else:
        current = set(initial_bitstrings)

    labels = op.paulis.to_labels()

    for _ in range(n_steps):
        nxt = set()
        for b in current:
            for label in labels:
                nxt.add(apply_ix_string_to_bitstring(label, b))
        current = nxt

    return current

def reachable_bitstrings_accumulating(op, initial_bitstrings, n_steps):
    if isinstance(initial_bitstrings, str):
        current = {initial_bitstrings}
    else:
        current = set(initial_bitstrings)

    labels = op.paulis.to_labels()

    for _ in range(n_steps):
        new = set()
        for b in current:
            for label in labels:
                new.add(apply_ix_string_to_bitstring(label, b))
        current |= new   # union with previous states

    return current

In [148]:
# # Example parameters
N = 4
F = 1
params = {
    14: dict(N=14, x=(14/30)**2, m_lat=10.0, g=1.0),
    16: dict(N=16, x=(16/30)**2, m_lat=10.0, g=1.0),
    18: dict(N=18, x=(18/30)**2, m_lat=10.0, g=1.0),
}
print(list(params[14]))
H = generateSchwingerHamiltonian(N = 4, x = (N/30)**2, lam=1000, l0= 3.1666666666, m_lat=10, g=1)
# params_ = params[14]
# H = generateSchwingerHamiltonian(params_['N'], params_['x'], lam = 1000,l0= 3.1666666666, m_lat= params_['m_lat'],g = params_['g'])
H_flip = bitflip_projection(H)

initial_state = '01' * (N//2)

['N', 'x', 'm_lat', 'g']


In [149]:
# from schwingermodel import siam_diagonal_bath_sparse_pauli, siam_quadratic_initial_state, siam_bitstring_initial_state
# K = 7
# U = 7.0
# t = 1.0
# V = 1.0

# H, eps, Vk = siam_diagonal_bath_sparse_pauli(K=K, U=U, t=t, V=V)
# H_flip = bitflip_projection(H)
# N = H.num_qubits
# print(f'{N} qubits in the algo')
# # not a bitstring...
# # initial_state = siam_quadratic_initial_state(K=K, U=U, t=t, V=V)

# psi0, occ = siam_bitstring_initial_state(K=K)

# initial_state = ''.join(str(b) for b in occ)

# Post-processing

# does not work yet

In [150]:
import numpy as np
import scipy.linalg
from qiskit.quantum_info import SparsePauliOp


def apply_pauli_to_bitstring(pauli_label: str, bitstring: str):
    """
    Apply a Pauli string to a computational basis bitstring.

    Returns
    -------
    phase : complex
        The phase acquired.
    out_bitstring : str
        The resulting computational basis bitstring.

    Conventions
    -----------
    - `pauli_label` is a Qiskit Pauli label string, e.g. "IXYZ".
    - `bitstring` is a standard computational basis bitstring, e.g. "0101".
    - We assume the displayed bitstring ordering matches the Pauli label ordering.
    """
    bits = list(bitstring)
    phase = 1.0 + 0.0j

    for i, p in enumerate(pauli_label):
        b = bits[i]

        if p == 'I':
            continue
        elif p == 'X':
            bits[i] = '1' if b == '0' else '0'
        elif p == 'Y':
            # Y|0> = i|1>,  Y|1> = -i|0>
            bits[i] = '1' if b == '0' else '0'
            phase *= 1j if b == '0' else -1j
        elif p == 'Z':
            # Z|0> = |0>, Z|1> = -|1>
            if b == '1':
                phase *= -1
        else:
            raise ValueError(f"Invalid Pauli character: {p}")

    return phase, ''.join(bits)


def project_hamiltonian_to_bitstring_subspace(H: SparsePauliOp, bitstrings):
    """
    Project a SparsePauliOp Hamiltonian into the subspace spanned by the
    given computational-basis bitstrings.

    Parameters
    ----------
    H : SparsePauliOp
        Full Hamiltonian on N qubits.
    bitstrings : sequence[str]
        Bitstrings defining the subspace basis.

    Returns
    -------
    H_sub : np.ndarray
        Dense projected Hamiltonian matrix in the bitstring basis.
    basis : list[str]
        The ordered basis actually used.
    """
    # Remove duplicates but preserve order
    basis = list(dict.fromkeys(bitstrings))
    m = len(basis)
    if m == 0:
        raise ValueError("bitstrings must not be empty")

    n = len(basis[0])
    if any(len(b) != n for b in basis):
        raise ValueError("All bitstrings must have the same length")

    basis_index = {b: i for i, b in enumerate(basis)}

    H_sub = np.zeros((m, m), dtype=complex)

    labels = H.paulis.to_labels()
    coeffs = H.coeffs

    # Matrix element construction:
    # For each column basis state |b_j>, act with each Pauli term.
    # If it lands on another basis state |b_i> inside the chosen subspace,
    # add coeff * phase to H_sub[i, j].
    for j, b_in in enumerate(basis):
        for label, coeff in zip(labels, coeffs):
            phase, b_out = apply_pauli_to_bitstring(label, b_in)
            i = basis_index.get(b_out)
            if i is not None:
                H_sub[i, j] += coeff * phase

    return H_sub, basis


def diagonalize_projected_hamiltonian(H_sub):
    """
    Diagonalize a projected Hamiltonian matrix.

    Uses scipy.linalg.eigh, assuming H_sub is Hermitian.
    """
    evals, evecs = scipy.linalg.eigh(H_sub)
    return evals, evecs

In [154]:
reachable = reachable_bitstrings_accumulating(H_flip, initial_state, 1)
bitstrings = sorted(reachable)

In [159]:
print(reachable)

{'0101', '0110', '1001', '0011'}


In [157]:
# H_full = generateSchwingerHamiltonian(N, x, lam, l0, m_lat, g)

H_sub, basis = project_hamiltonian_to_bitstring_subspace(H, bitstrings)

evals, evecs = diagonalize_projected_hamiltonian(H_sub)

# print("Basis:")
# print(basis)

# print("\nProjected Hamiltonian:")
# print(H_sub)

# print("\nEigenvalues:")
# print(evals)
print('vs. approximated via Krylov: ', min(evals))

vs. approximated via Krylov:  24.73219728731369


In [158]:

E0 = exact_ground_state_energy(H)
print("Exact ground state energy:", E0)
print('vs. approximated via Krylov: ', min(evals))

Exact ground state energy: 24.732197287264583
vs. approximated via Krylov:  24.73219728731369
